# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adnan-M123/flyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Refresh / Content Opportunity Scoring**

**Refresh / Content Opportunity Scoring**

*Why this one*

---

It aligns best with both my interests and the skills I'm currently developing. From my understanding, it combines data analysis with a practical outcome by helping prioritize which content should be reviewed and why. I like that it goes beyond finding patterns, it requires asking meaningful questions, understanding the factors behind the predictions, and producing recommendations that are transparent and explainable.



## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

My work improves the decision of **which content pages should be reviewed first** for actions such as refresh, expansion, protection, pruning, or monitoring. Since teams have limited time and resources, the goal is to prioritize the pages that are **most likely to benefit from human review** rather than attempting to review every page.

---

The ranked recommendations are **intended for SEO specialists, content strategists, or content editors**. They use the ranked queue, confidence scores, and reason codes to decide **which pages to investigate** and what type of action may be appropriate.

---

A false positive means reviewers spend time investigating a page that did not actually need attention, reducing the team's efficiency. A false negative means an important page is overlooked, potentially allowing its performance to continue declining or causing a missed opportunity for improvement. Since this is a decision-support system rather than an automated decision-maker, the cost of an **incorrect recommendation is primarily wasted review effort or missed opportunities**, not automatic changes to content. Human review remains part of the process.

## Framing check (framing-ml-problems skill)

*Why does data or ML help here, versus a plain rule or dashboard?*

A simple if-statement struggles here because many signals interact and shift over time — impressions, CTR, position, freshness, and engagement don't move independently, and their combined pattern is too tangled to hand-write reliably across thousands of pages. This isn't just a hunch: the starter pipeline's own results show a learned model clearly beating a fixed rule baseline (baseline precision@50 = 0.240 vs. random forest precision@50 = 0.740), which is real evidence that the pattern is there but too messy for an if-statement to capture on its own.

**Task type mapping**

| My question | Task type | Target | Metric |
|---|---|---|---|
| "Which pages should be reviewed first?" | Ranking / scoring | a priority score per page | precision@K (e.g. precision@50) |

**One-paragraph frame**

For SEO specialists and content strategists, deciding which pages to review first for refresh, expansion, protection, pruning, or monitoring, we will build a ranked priority queue from observable search and engagement signals (impressions, clicks, CTR, position, freshness, engagement), scoring priority for review, measured by precision@K (e.g. precision@50). A wrong call costs wasted reviewer time on a false positive, or a missed decline/opportunity on a false negative. A plain rule isn't enough because these signals interact and shift over time — the starter pipeline shows a learned model clearly beating a fixed rule (precision@50: 0.240 baseline vs 0.740 random forest). We will claim only observed, directional, and decision-support results.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [6]:
import pandas as pd
import os

# Colab needs the repo cloned first, since the badge only pulls this notebook,
# not the data folder next to it. Skip the clone if already running locally.
if not os.path.exists('flyrank-internship'):
    !git clone https://github.com/Adnan-M123/flyrank-internship.git

csv_path = 'flyrank-internship/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(csv_path)
print(f'raw rows: {len(df):,}, columns: {len(df.columns)}')

# Same filter/dedupe rule the starter pipeline uses (guide section 5),
# so the numbers below describe the same population the model will score.
df_f = (
    df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)]
    .drop_duplicates(subset='content_id')
)
print(f'rows after filter (impressions_90d > 0, content_age_days >= 90, deduped): {len(df_f):,}')
print()

# 1. How much of the inventory is actually declining right now?
declining_pct = (df_f['trend_direction'] == 'down').mean() * 100
print(f'declining pages (trend_direction == "down"): {declining_pct:.1f}%')

# 2. Of the pages that get real search traffic, how many are under-capturing clicks
#    for their position? (ctr is stored as a percent, so <0.5 means under 0.5%.)
low_ctr_visible = (
    (df_f['impressions_90d'] >= 500)
    & (df_f['avg_position'] > 0) & (df_f['avg_position'] <= 20)
    & (df_f['ctr'] < 0.5)
).mean() * 100
print(f'low-CTR visible pages (pos 1-20, ctr < 0.5%, >=500 impressions): {low_ctr_visible:.1f}%')

# 3. General scale, so a reader can trust these percentages aren't off a handful of rows.
print(f'median 90d impressions: {df_f["impressions_90d"].median():.0f}')
print(f'unique clients represented: {df_f["client_id"].nunique()}')

raw rows: 30,000, columns: 44
rows after filter (impressions_90d > 0, content_age_days >= 90, deduped): 30,000

declining pages (trend_direction == "down"): 54.2%
low-CTR visible pages (pos 1-20, ctr < 0.5%, >=500 impressions): 32.5%
median 90d impressions: 731
unique clients represented: 32


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.